# Lab 08 — Build a Multi-Agent System with the Gemini Agent SDK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-08-build-a-multi-agent-system-with-the-gemini-agent-sdk/lab-08-build-a-multi-agent-system-with-the-gemini-agent-sdk.ipynb)

**Topic:** 4 — Multi-Agent System Development with Gemini Agent SDK

**Objective:** Design collaborative agents with the Google Gemini Agent SDK (ADK)

Express the same multi-agent pattern in Google's ecosystem. Building the equivalent system twice separates the portable concepts — agents, tools, routing — from one vendor's API surface.

Full step-by-step instructions are in the Learner Guide.


In [ ]:
!pip install -q google-adk python-dotenv


Get a Gemini key from Google AI Studio: <https://aistudio.google.com/apikey>.

`GOOGLE_GENAI_USE_VERTEXAI=FALSE` tells ADK to use the AI Studio API rather than Vertex AI — omit it and you will get confusing authentication errors.


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("GOOGLE_API_KEY")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("Keys set:", [v for v in ["GOOGLE_API_KEY"] if os.environ.get(v)])


## 1. Two plain Python functions as tools

ADK reads the type hints and docstring to build the schema — no decorator is needed, unlike the OpenAI SDK's `@function_tool`.

Returning a dict with an explicit `status` key is the ADK convention. It gives the model an unambiguous signal about success or failure instead of making it infer from prose.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo


def get_weather(city: str) -> dict:
    """Retrieve the current weather report for a specified city.

    Args:
        city: The name of the city, for example "Singapore".

    Returns:
        A dict with a 'status' key and either a 'report' or 'error_message'.
    """
    readings = {
        "singapore": "31degC with thundery showers and 84% humidity.",
        "london": "12degC and overcast.",
        "tokyo": "18degC and clear.",
    }
    report = readings.get(city.strip().lower())
    if report is None:
        return {
            "status": "error",
            "error_message": f"No weather data available for '{city}'.",
        }
    return {"status": "success", "report": f"The weather in {city} is {report}"}


def get_time(city: str) -> dict:
    """Return the current local time in a specified city.

    Args:
        city: The name of the city, for example "Singapore".

    Returns:
        A dict with a 'status' key and either a 'report' or 'error_message'.
    """
    zones = {
        "singapore": "Asia/Singapore",
        "london": "Europe/London",
        "tokyo": "Asia/Tokyo",
    }
    zone = zones.get(city.strip().lower())
    if zone is None:
        return {
            "status": "error",
            "error_message": f"No timezone information for '{city}'.",
        }
    now = datetime.now(ZoneInfo(zone))
    return {
        "status": "success",
        "report": f"The current time in {city} is {now:%Y-%m-%d %H:%M:%S %Z}.",
    }


print(get_time("Tokyo"))
print(get_weather("Mars"))


## 2. The coordinator and its sub-agents

`description` and `instruction` do different jobs, and confusing them is the most common ADK mistake. The **description** is read by a *coordinator* deciding whether to route here — it is the discoverability text. The **instruction** is read by *this agent* when it runs — it is the behaviour text. An agent with a vague description will never be routed to, no matter how good its instruction is.

`name` must be a valid Python identifier: lowercase with underscores, no spaces.

The coordinator holds no tools of its own — its job is routing, exactly like the triage agent in Lab 06. A factory function is used because an agent instance may have only one parent, so do not attach the same sub-agent object to two coordinators.


In [ ]:
from google.adk.agents import Agent

MODEL = "gemini-2.0-flash"
APP_NAME = "multi_agent_lab"
USER_ID = "learner-1"


def build_coordinator(model: str = MODEL) -> Agent:
    """Construct the coordinator and its specialists for a given model."""
    weather = Agent(
        name="weather_agent",
        model=model,
        description="Answers questions about current weather conditions in a city.",
        instruction=(
            "You are a weather specialist. Use the get_weather tool to answer "
            "questions about conditions in a city. If the tool returns status "
            "'error', relay the error_message to the user rather than guessing."
        ),
        tools=[get_weather],
    )
    time_specialist = Agent(
        name="time_agent",
        model=model,
        description="Answers questions about the current local time in a city.",
        instruction=(
            "You are a timekeeping specialist. Use the get_time tool to report "
            "the local time in a city. If the tool returns status 'error', "
            "relay the error_message rather than guessing."
        ),
        tools=[get_time],
    )
    return Agent(
        name="coordinator",
        model=model,
        description="Routes user requests to the correct specialist sub-agent.",
        instruction=(
            "You are a coordinator. You do not answer questions yourself. "
            "Delegate weather questions to weather_agent and time questions to "
            "time_agent. If a request covers both, handle them in turn. "
            "If neither applies, say the request is outside your scope."
        ),
        sub_agents=[weather, time_specialist],
    )


coordinator = build_coordinator()
print(coordinator.name, "->", [a.name for a in coordinator.sub_agents])


## 3. The runner and the session service

ADK separates the **runner** (executes the agent loop) from the **session service** (carries conversational state). Three details worth noting: the user message must be wrapped in a `types.Content`, `create_session` is a coroutine so it needs `await`, and `run_async` is an async generator you consume with `async for`. Both `Runner(...)` and `run_async(...)` take keyword arguments only.

`event.author` carries the name of the agent that produced the response — the ADK equivalent of `result.last_agent.name`.


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


async def ask(runner: Runner, session_id: str, question: str) -> tuple[str, str]:
    """Send one question and return (final_text, responding_agent_name)."""
    message = types.Content(role="user", parts=[types.Part(text=question)])

    final_text, author = "", ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text or ""
            author = event.author
    return final_text, author


session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
runner = Runner(
    agent=coordinator,
    app_name=APP_NAME,
    session_service=session_service,
)
print("session:", session.id)


> In a notebook the cell above uses top-level `await`, which Colab supports. In the `gemini_team.py` script the same calls sit inside `async def main()` and are driven by `asyncio.run(main())`.


## 4. Route three requests and confirm the routing

Expect `weather_agent`, then `time_agent`, then a graceful error for Mars. Because all three questions share one `session.id`, the session service carries context between them.


In [ ]:
questions = [
    "What is the weather in Singapore?",
    "What time is it in Tokyo?",
    "What is the weather on Mars?",
]

for question in questions:
    answer, author = await ask(runner, session.id, question)
    print(f"\nQ: {question}")
    print(f"   handled by: {author}")
    print(f"   {answer}")


## 5. Confirm the session carries context

A follow-up with no explicit city should resolve against the earlier turns.


In [ ]:
answer, author = await ask(runner, session.id, "And what time is it there?")
print(f"[{author}] {answer}")


## 6. The API differences, with the architecture unchanged

Record this comparison — you extend it in Lab 09 and it is assessable material.

| Concept | OpenAI Agents SDK | Google ADK |
|---|---|---|
| Package / import | `openai-agents` / `from agents import ...` | `google-adk` / `from google.adk.agents import ...` |
| Define an agent | `Agent(name, instructions, model, tools)` | `Agent(name, model, description, instruction, tools)` |
| Declare a tool | `@function_tool` decorator | Plain function; schema from hints + docstring |
| Compose a team | `handoffs=[...]` | `sub_agents=[...]` |
| Routing signal | `handoff_description` | `description` |
| Execute | `Runner.run_sync(agent, "text")` | `runner.run_async(user_id=..., session_id=..., new_message=Content)` |
| Conversation state | Input list you pass in | `SessionService` holds it |
| Who answered | `result.last_agent.name` | `event.author` |
| Structured output | `output_type=PydanticModel` | `output_schema=PydanticModel` |

The architecture — a routing coordinator over narrow specialists with non-overlapping tools — is identical. Only the API surface changes, which is precisely the portable-concepts point of building it twice.

ADK also ships a web inspector that visualises the routing: run `adk web` from the parent folder of your agent package (locally, not in Colab).
